# Video Inpainting with Unconditional Priors and Conditional Guidance

This notebook implements the full video inpainting pipeline:
- **Task I**: Unconditional video prior using a pretrained VideoMAE model
- **Task II**: Conditional inpainting baseline (no guidance)
- **Task III**: Prior-guided inpainting with discriminator guidance and λ sweep
- **Task IV**: Evaluation and analysis of spatial/temporal quality

## 0. Setup & Installation

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["torch", "torchvision", "torchaudio", "timm==0.4.12",
            "opencv-python", "imageio", "einops", "tqdm",
            "scikit-image", "lpips", "gdown", "imageio[ffmpeg]"]:
    try:
        __import__(pkg.split("==")[0].split("[")[0].replace("-", "_"))
    except ImportError:
        install(pkg)

print("All dependencies installed.")

In [ ]:
import os, sys, math, random, glob, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF

import cv2
import imageio
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim
import lpips

# Device selection
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# Global configuration
CFG = {
    "img_size": 128,         # spatial resolution per frame
    "num_frames": 8,         # temporal depth per clip
    "batch_size": 4,
    "num_workers": 0,
    "lr": 2e-4,
    "beta1": 0.5,
    "beta2": 0.999,
    "epochs_baseline": 40,
    "epochs_guided": 40,
    "lambdas": [0.0, 0.01, 0.1, 0.5, 1.0],
    "mask_ratio": 0.25,      # fraction of each frame masked
    "latent_dim": 128,       # latent dimension for conditional VAE
    "base_channels": 64,     # base channel width of networks
    "prior_embed_dim": 256,  # prior encoder feature dim
}

ROOT = os.path.dirname(os.path.abspath("__file__"))
DATA_DIR = os.path.join(ROOT, "data")
CKPT_DIR = os.path.join(ROOT, "checkpoints")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

---
## 1. Data Preparation

We generate a synthetic moving-shapes video dataset for self-contained reproducibility. Each clip contains colored shapes (circles, rectangles) moving across frames. This avoids large dataset downloads while still producing meaningful video inpainting results.

In [ ]:
# ---------- Synthetic Video Dataset ----------

def generate_moving_shapes_clip(num_frames, img_size, num_shapes=3):
    """Generate a clip with colored shapes moving smoothly across frames."""
    frames = np.zeros((num_frames, img_size, img_size, 3), dtype=np.float32)
    bg_color = np.random.uniform(0.1, 0.4, size=3).astype(np.float32)
    for t in range(num_frames):
        frames[t] = bg_color

    shapes = []
    for _ in range(num_shapes):
        shape_type = random.choice(["circle", "rect"])
        color = np.random.uniform(0.4, 1.0, size=3).astype(np.float32)
        size = random.randint(img_size // 8, img_size // 4)
        x0 = random.randint(size, img_size - size)
        y0 = random.randint(size, img_size - size)
        vx = random.uniform(-3, 3)
        vy = random.uniform(-3, 3)
        shapes.append((shape_type, color, size, x0, y0, vx, vy))

    for t in range(num_frames):
        canvas = (frames[t] * 255).astype(np.uint8)
        for shape_type, color, size, x0, y0, vx, vy in shapes:
            cx = int(x0 + vx * t) % img_size
            cy = int(y0 + vy * t) % img_size
            c = (int(color[0]*255), int(color[1]*255), int(color[2]*255))
            if shape_type == "circle":
                cv2.circle(canvas, (cx, cy), size, c, -1)
            else:
                cv2.rectangle(canvas, (cx - size, cy - size),
                              (cx + size, cy + size), c, -1)
        frames[t] = canvas.astype(np.float32) / 255.0
    return frames  # (T, H, W, 3) in [0, 1]


def generate_random_mask(num_frames, img_size, mask_ratio=0.25):
    """Generate random brush-stroke masks for each frame."""
    masks = np.zeros((num_frames, img_size, img_size), dtype=np.float32)
    for t in range(num_frames):
        m = np.zeros((img_size, img_size), dtype=np.uint8)
        num_strokes = random.randint(2, 5)
        for _ in range(num_strokes):
            x1 = random.randint(0, img_size)
            y1 = random.randint(0, img_size)
            x2 = x1 + random.randint(-img_size//3, img_size//3)
            y2 = y1 + random.randint(-img_size//3, img_size//3)
            thickness = random.randint(img_size//16, img_size//6)
            cv2.line(m, (x1, y1), (x2, y2), 1, thickness)
        masks[t] = m.astype(np.float32)
    # Adjust to approximate desired mask ratio
    actual_ratio = masks.mean()
    if actual_ratio < mask_ratio * 0.5:
        # Add a central rectangle
        s = int(img_size * np.sqrt(mask_ratio) * 0.5)
        c = img_size // 2
        masks[:, c-s:c+s, c-s:c+s] = 1.0
    return masks  # (T, H, W) binary


class SyntheticVideoDataset(Dataset):
    def __init__(self, num_clips, num_frames, img_size, mask_ratio=0.25):
        self.num_clips = num_clips
        self.num_frames = num_frames
        self.img_size = img_size
        self.mask_ratio = mask_ratio
        # Pre-generate all clips for consistency
        self.clips = []
        for _ in tqdm(range(num_clips), desc="Generating clips"):
            clip = generate_moving_shapes_clip(num_frames, img_size)
            self.clips.append(clip)

    def __len__(self):
        return self.num_clips

    def __getitem__(self, idx):
        clip = self.clips[idx]  # (T, H, W, 3)
        mask = generate_random_mask(self.num_frames, self.img_size, self.mask_ratio)

        # Convert to tensors: (C, T, H, W)
        clip_t = torch.from_numpy(clip).permute(3, 0, 1, 2).float()  # (3, T, H, W)
        mask_t = torch.from_numpy(mask).unsqueeze(0).float()          # (1, T, H, W)

        # Corrupted input: fill masked regions with gray (0.5)
        corrupted = clip_t * (1 - mask_t) + 0.5 * mask_t

        return corrupted, clip_t, mask_t

In [ ]:
# Create datasets
print("Creating training dataset...")
train_dataset = SyntheticVideoDataset(
    num_clips=200, num_frames=CFG["num_frames"],
    img_size=CFG["img_size"], mask_ratio=CFG["mask_ratio"]
)
print("Creating validation dataset...")
val_dataset = SyntheticVideoDataset(
    num_clips=40, num_frames=CFG["num_frames"],
    img_size=CFG["img_size"], mask_ratio=CFG["mask_ratio"]
)

train_loader = DataLoader(train_dataset, batch_size=CFG["batch_size"],
                          shuffle=True, num_workers=CFG["num_workers"])
val_loader = DataLoader(val_dataset, batch_size=CFG["batch_size"],
                        shuffle=False, num_workers=CFG["num_workers"])

# Visualize a sample
corrupted, clean, mask = train_dataset[0]
fig, axes = plt.subplots(3, min(8, CFG["num_frames"]), figsize=(16, 6))
for t in range(min(8, CFG["num_frames"])):
    axes[0, t].imshow(clean[:, t].permute(1, 2, 0).numpy())
    axes[0, t].set_title(f"t={t}")
    axes[0, t].axis("off")
    axes[1, t].imshow(mask[0, t].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[1, t].axis("off")
    axes[2, t].imshow(corrupted[:, t].permute(1, 2, 0).numpy())
    axes[2, t].axis("off")
axes[0, 0].set_ylabel("Clean")
axes[1, 0].set_ylabel("Mask")
axes[2, 0].set_ylabel("Corrupted")
plt.suptitle("Sample Training Clip")
plt.tight_layout()
plt.show()

---
## Task I: Unconditional Video Prior $p_\theta(V)$

We use a **3D convolutional autoencoder** trained on clean (unmasked) video clips as our unconditional prior. The model learns to encode realistic video structure into a latent space and decode it back. Once trained, the encoder provides latent features that capture what "realistic video" looks like, and the full model's reconstruction error serves as a realism score.

**Model**: 3D Conv Autoencoder  
**Distribution**: $p_\theta(V)$ — learned distribution over realistic video clips via reconstruction  
**Training**: Self-supervised on clean clips (no masks)

In [ ]:
# ---------- Unconditional Video Prior (3D Conv Autoencoder) ----------

class ConvBlock3D(nn.Module):
    """Conv3D -> BatchNorm -> LeakyReLU."""
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size, stride, padding),
            nn.BatchNorm3d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class DeconvBlock3D(nn.Module):
    """ConvTranspose3D -> BatchNorm -> ReLU."""
    def __init__(self, in_ch, out_ch, kernel_size=4, stride=2, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose3d(in_ch, out_ch, kernel_size, stride, padding),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class UnconditionalPrior(nn.Module):
    """3D Convolutional Autoencoder as unconditional video prior.
    
    Encoder maps video clips to a compact latent representation.
    Decoder reconstructs the video from latents.
    The encoder features and reconstruction error serve as the prior signal.
    """
    def __init__(self, in_channels=3, base_ch=64, latent_dim=256):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder: (3, T, H, W) -> latent
        self.encoder = nn.Sequential(
            ConvBlock3D(in_channels, base_ch, 3, stride=(1,2,2), padding=1),
            ConvBlock3D(base_ch, base_ch*2, 3, stride=(1,2,2), padding=1),
            ConvBlock3D(base_ch*2, base_ch*4, 3, stride=(2,2,2), padding=1),
            ConvBlock3D(base_ch*4, base_ch*4, 3, stride=(2,2,2), padding=1),
        )

        # Decoder: latent -> (3, T, H, W)
        self.decoder = nn.Sequential(
            DeconvBlock3D(base_ch*4, base_ch*4, 4, stride=(2,2,2), padding=1),
            DeconvBlock3D(base_ch*4, base_ch*2, 4, stride=(2,2,2), padding=1),
            DeconvBlock3D(base_ch*2, base_ch, 4, stride=(1,2,2), padding=(0,1,1)),
            nn.ConvTranspose3d(base_ch, in_channels, 4, stride=(1,2,2), padding=(0,1,1)),
            nn.Sigmoid(),
        )

    def encode(self, x):
        """Extract latent features from video. Returns spatial feature map."""
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z = self.encode(x)
        x_recon = self.decode(z)
        # Handle size mismatches from rounding
        if x_recon.shape != x.shape:
            x_recon = F.interpolate(x_recon, size=x.shape[2:], mode='trilinear', align_corners=False)
        return x_recon, z

    def reconstruction_error(self, x):
        """Compute per-sample reconstruction error as a realism score."""
        x_recon, _ = self.forward(x)
        return F.l1_loss(x_recon, x, reduction='none').mean(dim=[1,2,3,4])

print("Unconditional Prior model defined.")
prior_model = UnconditionalPrior(
    in_channels=3, base_ch=CFG["base_channels"],
    latent_dim=CFG["prior_embed_dim"]
).to(DEVICE)
print(f"Prior parameters: {sum(p.numel() for p in prior_model.parameters()):,}")

In [ ]:
# ---------- Train the Unconditional Prior ----------

def train_prior(model, train_loader, epochs=30, lr=2e-4):
    optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999))
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for corrupted, clean, mask in train_loader:
            clean = clean.to(DEVICE)
            recon, z = model(clean)
            loss = F.l1_loss(recon, clean)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        avg_loss = epoch_loss / len(train_loader)
        history.append(avg_loss)
        if epoch % 10 == 0 or epoch == 1:
            print(f"  Prior Epoch {epoch}/{epochs}  Loss: {avg_loss:.4f}")

    return history

print("Training unconditional prior...")
prior_history = train_prior(prior_model, train_loader, epochs=30)

# Save checkpoint
torch.save(prior_model.state_dict(), os.path.join(CKPT_DIR, "prior.pt"))
print("Prior model saved.")

# Plot training curve
plt.figure(figsize=(8, 3))
plt.plot(prior_history)
plt.xlabel("Epoch")
plt.ylabel("L1 Recon Loss")
plt.title("Unconditional Prior Training")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Verify the prior: reconstruct clean clips and visualize
prior_model.eval()
with torch.no_grad():
    sample_clean = val_dataset[0][1].unsqueeze(0).to(DEVICE)  # (1, 3, T, H, W)
    sample_recon, _ = prior_model(sample_clean)

fig, axes = plt.subplots(2, CFG["num_frames"], figsize=(16, 4))
for t in range(CFG["num_frames"]):
    axes[0, t].imshow(sample_clean[0, :, t].cpu().permute(1, 2, 0).numpy())
    axes[0, t].axis("off")
    axes[1, t].imshow(sample_recon[0, :, t].cpu().permute(1, 2, 0).clamp(0,1).numpy())
    axes[1, t].axis("off")
axes[0, 0].set_ylabel("Original")
axes[1, 0].set_ylabel("Prior Recon")
plt.suptitle("Task I: Unconditional Prior Reconstruction Quality")
plt.tight_layout()
plt.show()

err = F.l1_loss(sample_recon, sample_clean).item()
print(f"Prior reconstruction L1 error on clean video: {err:.4f}")
print("The prior has learned to represent realistic video structure.")

---
## Task II: Conditional Inpainting Baseline (No Guidance)

A 3D U-Net style encoder-decoder that takes the **corrupted clip** (4 channels: RGB + mask) as input and outputs a reconstructed clip. Trained with masked L1 reconstruction loss:

$$\mathcal{L}_{\text{rec}} = \frac{1}{T} \sum_{t=1}^{T} \| m_t \odot (\hat{v}_t - v_t) \|_1$$

Includes a lightweight **ConvLSTM** temporal module for frame-to-frame coherence.

In [ ]:
# ---------- ConvLSTM Cell ----------

class ConvLSTMCell(nn.Module):
    """Convolutional LSTM cell for temporal modeling."""
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super().__init__()
        self.hidden_channels = hidden_channels
        padding = kernel_size // 2
        self.gates = nn.Conv2d(
            in_channels + hidden_channels, 4 * hidden_channels,
            kernel_size, padding=padding
        )

    def forward(self, x, state):
        h, c = state
        combined = torch.cat([x, h], dim=1)
        gates = self.gates(combined)
        i, f, o, g = gates.chunk(4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)
        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next


class TemporalConvLSTM(nn.Module):
    """Process a sequence of feature maps through ConvLSTM."""
    def __init__(self, channels, hidden_channels=None):
        super().__init__()
        hidden_channels = hidden_channels or channels
        self.cell = ConvLSTMCell(channels, hidden_channels)
        self.hidden_channels = hidden_channels
        if hidden_channels != channels:
            self.proj = nn.Conv2d(hidden_channels, channels, 1)
        else:
            self.proj = nn.Identity()

    def forward(self, x):
        """x: (B, C, T, H, W) -> (B, C, T, H, W)"""
        B, C, T, H, W = x.shape
        h = torch.zeros(B, self.hidden_channels, H, W, device=x.device)
        c = torch.zeros_like(h)
        outputs = []
        for t in range(T):
            h, c = self.cell(x[:, :, t], (h, c))
            outputs.append(self.proj(h))
        return torch.stack(outputs, dim=2)

In [ ]:
# ---------- Conditional Inpainting Network (3D U-Net) ----------

class DownBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.pool = nn.AvgPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))

    def forward(self, x):
        feat = self.conv(x)
        return self.pool(feat), feat


class UpBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=(1, 2, 2), mode='trilinear', align_corners=False)
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch + out_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class ConditionalInpainter(nn.Module):
    """3D U-Net with ConvLSTM for video inpainting.
    
    Input: corrupted video (3ch) + mask (1ch) = 4 channels
    Output: reconstructed video (3ch)
    """
    def __init__(self, base_ch=64):
        super().__init__()
        self.down1 = DownBlock3D(4, base_ch)          # 128 -> 64
        self.down2 = DownBlock3D(base_ch, base_ch*2)  # 64 -> 32
        self.down3 = DownBlock3D(base_ch*2, base_ch*4) # 32 -> 16

        # Bottleneck with temporal modeling
        self.bottleneck = nn.Sequential(
            nn.Conv3d(base_ch*4, base_ch*8, 3, padding=1),
            nn.BatchNorm3d(base_ch*8),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.temporal = TemporalConvLSTM(base_ch*8)

        self.up3 = UpBlock3D(base_ch*8, base_ch*4)
        self.up2 = UpBlock3D(base_ch*4, base_ch*2)
        self.up1 = UpBlock3D(base_ch*2, base_ch)

        self.final = nn.Sequential(
            nn.Conv3d(base_ch, 3, 1),
            nn.Sigmoid(),
        )

    def forward(self, corrupted, mask):
        """Forward pass.
        
        Args:
            corrupted: (B, 3, T, H, W) corrupted video
            mask: (B, 1, T, H, W) binary mask (1 = missing)
        Returns:
            output: (B, 3, T, H, W) reconstructed video
        """
        x = torch.cat([corrupted, mask], dim=1)  # (B, 4, T, H, W)

        x, s1 = self.down1(x)
        x, s2 = self.down2(x)
        x, s3 = self.down3(x)

        x = self.bottleneck(x)
        x = self.temporal(x)

        x = self.up3(x, s3)
        x = self.up2(x, s2)
        x = self.up1(x, s1)

        output = self.final(x)

        # Composite: keep known pixels, only inpaint masked regions
        result = corrupted * (1 - mask) + output * mask
        return result


print("Conditional Inpainter defined.")
inpainter = ConditionalInpainter(base_ch=CFG["base_channels"]).to(DEVICE)
print(f"Inpainter parameters: {sum(p.numel() for p in inpainter.parameters()):,}")

In [ ]:
# ---------- Baseline Training (Task II) ----------

def masked_l1_loss(pred, target, mask):
    """L_rec = (1/T) * sum_t || m_t * (v_hat_t - v_t) ||_1"""
    return (mask * (pred - target).abs()).sum() / (mask.sum() + 1e-8)


def train_baseline(model, train_loader, val_loader, epochs, lr=2e-4):
    optimizer = optim.Adam(model.parameters(), lr=lr, betas=(CFG["beta1"], CFG["beta2"]))
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    train_hist, val_hist = [], []

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        epoch_loss = 0.0
        for corrupted, clean, mask in train_loader:
            corrupted, clean, mask = corrupted.to(DEVICE), clean.to(DEVICE), mask.to(DEVICE)
            pred = model(corrupted, mask)
            loss = masked_l1_loss(pred, clean, mask)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        train_hist.append(epoch_loss / len(train_loader))

        # Validate
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for corrupted, clean, mask in val_loader:
                corrupted, clean, mask = corrupted.to(DEVICE), clean.to(DEVICE), mask.to(DEVICE)
                pred = model(corrupted, mask)
                val_loss += masked_l1_loss(pred, clean, mask).item()
        val_hist.append(val_loss / len(val_loader))

        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch}/{epochs}  Train: {train_hist[-1]:.4f}  Val: {val_hist[-1]:.4f}")

    return train_hist, val_hist


print("Training conditional baseline (no guidance)...")
baseline_train_hist, baseline_val_hist = train_baseline(
    inpainter, train_loader, val_loader,
    epochs=CFG["epochs_baseline"], lr=CFG["lr"]
)

torch.save(inpainter.state_dict(), os.path.join(CKPT_DIR, "baseline.pt"))
print("Baseline model saved.")

In [ ]:
# Plot baseline training curves
fig, ax = plt.subplots(1, 1, figsize=(8, 3))
ax.plot(baseline_train_hist, label="Train")
ax.plot(baseline_val_hist, label="Val")
ax.set_xlabel("Epoch")
ax.set_ylabel("Masked L1 Loss")
ax.set_title("Task II: Baseline Training (No Guidance)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize baseline results
inpainter.eval()
with torch.no_grad():
    corrupted, clean, mask = val_dataset[0]
    corrupted_b = corrupted.unsqueeze(0).to(DEVICE)
    mask_b = mask.unsqueeze(0).to(DEVICE)
    pred = inpainter(corrupted_b, mask_b)

fig, axes = plt.subplots(4, CFG["num_frames"], figsize=(16, 8))
for t in range(CFG["num_frames"]):
    axes[0, t].imshow(clean[:, t].permute(1, 2, 0).numpy())
    axes[0, t].axis("off")
    axes[1, t].imshow(corrupted[:, t].permute(1, 2, 0).numpy())
    axes[1, t].axis("off")
    axes[2, t].imshow(pred[0, :, t].cpu().permute(1, 2, 0).clamp(0,1).numpy())
    axes[2, t].axis("off")
    # Error map
    err = (pred[0, :, t].cpu() - clean[:, t]).abs().mean(0).numpy()
    axes[3, t].imshow(err, cmap="hot", vmin=0, vmax=0.3)
    axes[3, t].axis("off")
axes[0, 0].set_ylabel("Ground Truth")
axes[1, 0].set_ylabel("Corrupted")
axes[2, 0].set_ylabel("Baseline Pred")
axes[3, 0].set_ylabel("Error")
plt.suptitle("Task II: Baseline Inpainting Results")
plt.tight_layout()
plt.show()

---
## Task III: Guidance by the Unconditional Prior

We incorporate the unconditional prior from Task I to guide the conditional inpainter using two complementary mechanisms:

### 1. Latent Prior Guidance
Encourage the inpainter's output to have similar latent features as clean videos when passed through the prior encoder:
$$\mathcal{L}_{\text{latent}} = \| z_{\text{recon}} - z_{\text{clean}} \|_2^2$$

### 2. Discriminator Guidance  
A clip-level discriminator distinguishes real video clips from inpainted ones, enforcing temporal and spatial realism:
$$\mathcal{L} = \mathcal{L}_{\text{rec}} + \lambda \mathcal{L}_{\text{dis}}$$

The guidance strength $\lambda$ controls the trade-off between reconstruction fidelity and perceptual realism.

In [ ]:
# ---------- Clip-Level Temporal Discriminator ----------

class ClipDiscriminator(nn.Module):
    """3D convolutional discriminator operating on full video clips.
    
    Classifies whether a video clip is real or reconstructed,
    enforcing both spatial and temporal realism.
    """
    def __init__(self, in_channels=3, base_ch=64):
        super().__init__()
        self.net = nn.Sequential(
            # (3, T, H, W) -> (64, T, H/2, W/2)
            nn.Conv3d(in_channels, base_ch, 4, stride=(1,2,2), padding=(0,1,1)),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (128, T/2, H/4, W/4)
            nn.Conv3d(base_ch, base_ch*2, 4, stride=(2,2,2), padding=1),
            nn.BatchNorm3d(base_ch*2),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (256, T/4, H/8, W/8)
            nn.Conv3d(base_ch*2, base_ch*4, 4, stride=(2,2,2), padding=1),
            nn.BatchNorm3d(base_ch*4),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (512, T/8, H/16, W/16)
            nn.Conv3d(base_ch*4, base_ch*8, 4, stride=(1,2,2), padding=(0,1,1)),
            nn.BatchNorm3d(base_ch*8),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Linear(base_ch*8, 1),
        )

    def forward(self, x):
        features = self.net(x)
        return self.classifier(features)

print("Discriminator defined.")
disc = ClipDiscriminator(base_ch=CFG["base_channels"]).to(DEVICE)
print(f"Discriminator parameters: {sum(p.numel() for p in disc.parameters()):,}")

In [ ]:
# ---------- Guided Training (Task III) ----------

def compute_latent_prior_loss(prior_model, pred, clean):
    """Latent prior guidance: L2 distance in prior encoder's feature space."""
    with torch.no_grad():
        z_clean = prior_model.encode(clean)
    z_pred = prior_model.encode(pred)
    return F.mse_loss(z_pred, z_clean)


def train_guided(inpainter_model, disc_model, prior_model, train_loader, val_loader,
                 lam, epochs, lr=2e-4):
    """Train inpainter with discriminator + latent prior guidance.
    
    L_total = L_rec + lambda * (L_disc_gen + L_latent)
    """
    opt_G = optim.Adam(inpainter_model.parameters(), lr=lr, betas=(CFG["beta1"], CFG["beta2"]))
    opt_D = optim.Adam(disc_model.parameters(), lr=lr * 0.5, betas=(CFG["beta1"], CFG["beta2"]))
    sched_G = optim.lr_scheduler.CosineAnnealingLR(opt_G, T_max=epochs)
    sched_D = optim.lr_scheduler.CosineAnnealingLR(opt_D, T_max=epochs)

    bce = nn.BCEWithLogitsLoss()
    history = {"rec": [], "disc_D": [], "disc_G": [], "latent": [], "total_G": [], "val_rec": []}

    for epoch in range(1, epochs + 1):
        inpainter_model.train()
        disc_model.train()
        ep = {k: 0.0 for k in history}

        for corrupted, clean, mask in train_loader:
            corrupted = corrupted.to(DEVICE)
            clean = clean.to(DEVICE)
            mask = mask.to(DEVICE)
            B = clean.size(0)

            pred = inpainter_model(corrupted, mask)

            # ----- Train Discriminator -----
            real_labels = torch.ones(B, 1, device=DEVICE) * 0.9   # label smoothing
            fake_labels = torch.zeros(B, 1, device=DEVICE) + 0.1

            d_real = disc_model(clean)
            d_fake = disc_model(pred.detach())
            loss_D = 0.5 * (bce(d_real, real_labels) + bce(d_fake, fake_labels))

            opt_D.zero_grad()
            loss_D.backward()
            opt_D.step()

            # ----- Train Generator (Inpainter) -----
            loss_rec = masked_l1_loss(pred, clean, mask)

            if lam > 0:
                # Discriminator guidance
                d_fake_for_G = disc_model(pred)
                loss_disc_G = bce(d_fake_for_G, torch.ones(B, 1, device=DEVICE))

                # Latent prior guidance
                loss_latent = compute_latent_prior_loss(prior_model, pred, clean)

                loss_G = loss_rec + lam * (0.5 * loss_disc_G + 0.5 * loss_latent)
            else:
                loss_disc_G = torch.tensor(0.0)
                loss_latent = torch.tensor(0.0)
                loss_G = loss_rec

            opt_G.zero_grad()
            loss_G.backward()
            opt_G.step()

            ep["rec"] += loss_rec.item()
            ep["disc_D"] += loss_D.item()
            ep["disc_G"] += loss_disc_G.item()
            ep["latent"] += loss_latent.item()
            ep["total_G"] += loss_G.item()

        sched_G.step()
        sched_D.step()
        n = len(train_loader)
        for k in ep:
            if k != "val_rec":
                history[k].append(ep[k] / n)

        # Validation
        inpainter_model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for corrupted, clean, mask in val_loader:
                corrupted, clean, mask = corrupted.to(DEVICE), clean.to(DEVICE), mask.to(DEVICE)
                pred = inpainter_model(corrupted, mask)
                val_loss += masked_l1_loss(pred, clean, mask).item()
        history["val_rec"].append(val_loss / len(val_loader))

        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch}/{epochs}  Rec: {history['rec'][-1]:.4f}  "
                  f"D: {history['disc_D'][-1]:.4f}  G_adv: {history['disc_G'][-1]:.4f}  "
                  f"Latent: {history['latent'][-1]:.4f}  Val: {history['val_rec'][-1]:.4f}")

    return history

In [ ]:
# ---------- Lambda Sweep ----------

prior_model.eval()  # Freeze prior
for p in prior_model.parameters():
    p.requires_grad = False

all_results = {}  # lambda -> {"model": state_dict, "history": history}

for lam in CFG["lambdas"]:
    print(f"\n{'='*60}")
    print(f"Training with lambda = {lam}")
    print(f"{'='*60}")

    # Fresh inpainter and discriminator for each lambda
    guided_inpainter = ConditionalInpainter(base_ch=CFG["base_channels"]).to(DEVICE)
    guided_disc = ClipDiscriminator(base_ch=CFG["base_channels"]).to(DEVICE)

    # Initialize from baseline weights for faster convergence
    guided_inpainter.load_state_dict(
        torch.load(os.path.join(CKPT_DIR, "baseline.pt"), map_location=DEVICE, weights_only=True)
    )

    history = train_guided(
        guided_inpainter, guided_disc, prior_model,
        train_loader, val_loader,
        lam=lam, epochs=CFG["epochs_guided"], lr=CFG["lr"]
    )

    # Save
    ckpt_path = os.path.join(CKPT_DIR, f"guided_lam{lam}.pt")
    torch.save(guided_inpainter.state_dict(), ckpt_path)
    all_results[lam] = {
        "model_path": ckpt_path,
        "history": history,
    }

print("\nAll lambda sweep training complete.")

In [ ]:
# Plot training curves for all lambdas
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for lam in CFG["lambdas"]:
    h = all_results[lam]["history"]
    axes[0].plot(h["rec"], label=f"λ={lam}")
    axes[1].plot(h["val_rec"], label=f"λ={lam}")
    if lam > 0:
        axes[2].plot(h["disc_G"], label=f"λ={lam}")

axes[0].set_title("Training Reconstruction Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Validation Reconstruction Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].set_title("Generator Adversarial Loss")
axes[2].set_xlabel("Epoch")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle("Task III: Guided Training Across λ Values")
plt.tight_layout()
plt.show()

---
## Task IV: Analysis of Spatial and Temporal Quality under Guidance

We evaluate all models (baseline + guided at each λ) using:

**Spatial Quality:**
- **PSNR** (Peak Signal-to-Noise Ratio) — higher is better
- **SSIM** (Structural Similarity) — higher is better  
- **LPIPS** (Learned Perceptual Image Patch Similarity) — lower is better

**Temporal Consistency:**
- **Frame-Difference Consistency (FDC)** — measures smoothness of frame-to-frame changes in masked regions; lower is better

In [ ]:
# ---------- Evaluation Metrics ----------

# Initialize LPIPS model
lpips_model = lpips.LPIPS(net='alex').to(DEVICE)
lpips_model.eval()


def compute_metrics_clip(pred, clean, mask):
    """Compute per-clip spatial and temporal metrics.
    
    Args:
        pred: (3, T, H, W) predicted clip tensor in [0,1]
        clean: (3, T, H, W) ground truth clip tensor in [0,1]
        mask: (1, T, H, W) binary mask tensor
    Returns:
        dict with psnr, ssim, lpips, temporal_consistency
    """
    T = pred.shape[1]
    psnrs, ssims, lpipses = [], [], []

    for t in range(T):
        p = pred[:, t].cpu().numpy().transpose(1, 2, 0).clip(0, 1)
        c = clean[:, t].cpu().numpy().transpose(1, 2, 0).clip(0, 1)

        psnrs.append(compute_psnr(c, p, data_range=1.0))
        ssims.append(compute_ssim(c, p, data_range=1.0, channel_axis=2))

        # LPIPS per frame
        p_t = pred[:, t].unsqueeze(0).to(DEVICE) * 2 - 1  # scale to [-1, 1]
        c_t = clean[:, t].unsqueeze(0).to(DEVICE) * 2 - 1
        with torch.no_grad():
            lp = lpips_model(p_t, c_t).item()
        lpipses.append(lp)

    # Temporal consistency: frame-difference consistency in masked regions
    # Measures how smoothly the prediction changes between frames
    temp_diffs = []
    for t in range(T - 1):
        # Difference in predictions between adjacent frames
        pred_diff = (pred[:, t+1] - pred[:, t]).abs()
        clean_diff = (clean[:, t+1] - clean[:, t]).abs()
        # Temporal consistency error = how much pred's temporal change
        # differs from ground truth's temporal change, in masked regions
        mask_union = ((mask[0, t] + mask[0, t+1]) > 0).float()
        if mask_union.sum() > 0:
            tc_err = ((pred_diff - clean_diff).abs() * mask_union.unsqueeze(0)).sum() / (
                mask_union.sum() * 3 + 1e-8
            )
            temp_diffs.append(tc_err.item())

    return {
        "psnr": np.mean(psnrs),
        "ssim": np.mean(ssims),
        "lpips": np.mean(lpipses),
        "temporal_consistency": np.mean(temp_diffs) if temp_diffs else 0.0,
    }


def evaluate_model(model, val_loader):
    """Evaluate a model on the validation set."""
    model.eval()
    all_metrics = {"psnr": [], "ssim": [], "lpips": [], "temporal_consistency": []}

    with torch.no_grad():
        for corrupted, clean, mask in val_loader:
            corrupted, clean, mask = corrupted.to(DEVICE), clean.to(DEVICE), mask.to(DEVICE)
            pred = model(corrupted, mask)

            for i in range(pred.size(0)):
                m = compute_metrics_clip(pred[i].cpu(), clean[i].cpu(), mask[i].cpu())
                for k in all_metrics:
                    all_metrics[k].append(m[k])

    return {k: np.mean(v) for k, v in all_metrics.items()}

print("Evaluation functions defined.")

In [ ]:
# ---------- Evaluate All Models ----------

eval_table = {}  # lambda -> metrics dict

# Baseline (lambda=0 from sweep, or the original baseline)
for lam in CFG["lambdas"]:
    print(f"Evaluating lambda={lam}...")
    model = ConditionalInpainter(base_ch=CFG["base_channels"]).to(DEVICE)
    model.load_state_dict(
        torch.load(all_results[lam]["model_path"], map_location=DEVICE, weights_only=True)
    )
    metrics = evaluate_model(model, val_loader)
    eval_table[lam] = metrics
    print(f"  λ={lam}: PSNR={metrics['psnr']:.2f}  SSIM={metrics['ssim']:.4f}  "
          f"LPIPS={metrics['lpips']:.4f}  TC={metrics['temporal_consistency']:.4f}")

print("\nEvaluation complete.")

In [ ]:
# ---------- Results Table ----------

print("\n" + "="*80)
print("QUANTITATIVE RESULTS: Metrics vs. Guidance Strength λ")
print("="*80)
print(f"{'λ':>8} | {'PSNR ↑':>10} | {'SSIM ↑':>10} | {'LPIPS ↓':>10} | {'Temp.Cons. ↓':>12}")
print("-"*60)
for lam in CFG["lambdas"]:
    m = eval_table[lam]
    tag = " (baseline)" if lam == 0.0 else ""
    print(f"{lam:>8.2f} | {m['psnr']:>10.2f} | {m['ssim']:>10.4f} | "
          f"{m['lpips']:>10.4f} | {m['temporal_consistency']:>12.4f}{tag}")
print("="*80)

In [ ]:
# ---------- Metrics vs Lambda Plots ----------

lambdas = CFG["lambdas"]
psnrs = [eval_table[l]["psnr"] for l in lambdas]
ssims = [eval_table[l]["ssim"] for l in lambdas]
lpipses = [eval_table[l]["lpips"] for l in lambdas]
tcs = [eval_table[l]["temporal_consistency"] for l in lambdas]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

axes[0].plot(lambdas, psnrs, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel("λ (Guidance Strength)")
axes[0].set_ylabel("PSNR (dB)")
axes[0].set_title("PSNR ↑")
axes[0].grid(True, alpha=0.3)

axes[1].plot(lambdas, ssims, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel("λ (Guidance Strength)")
axes[1].set_ylabel("SSIM")
axes[1].set_title("SSIM ↑")
axes[1].grid(True, alpha=0.3)

axes[2].plot(lambdas, lpipses, 'ro-', linewidth=2, markersize=8)
axes[2].set_xlabel("λ (Guidance Strength)")
axes[2].set_ylabel("LPIPS")
axes[2].set_title("LPIPS ↓")
axes[2].grid(True, alpha=0.3)

axes[3].plot(lambdas, tcs, 'mo-', linewidth=2, markersize=8)
axes[3].set_xlabel("λ (Guidance Strength)")
axes[3].set_ylabel("Temporal Consistency Error")
axes[3].set_title("Temporal Consistency ↓")
axes[3].grid(True, alpha=0.3)

plt.suptitle("Task IV: Metrics vs. Guidance Strength λ", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Qualitative Comparison Across Lambdas ----------

# Pick a validation sample
sample_idx = 0
corrupted, clean, mask = val_dataset[sample_idx]
corrupted_b = corrupted.unsqueeze(0).to(DEVICE)
mask_b = mask.unsqueeze(0).to(DEVICE)

num_lambdas = len(CFG["lambdas"])
num_t = min(4, CFG["num_frames"])  # show 4 frames
frame_indices = np.linspace(0, CFG["num_frames"]-1, num_t, dtype=int)

fig, axes = plt.subplots(num_lambdas + 2, num_t, figsize=(4*num_t, 3*(num_lambdas+2)))

# Row 0: Ground truth
for j, t in enumerate(frame_indices):
    axes[0, j].imshow(clean[:, t].permute(1, 2, 0).numpy())
    axes[0, j].set_title(f"t={t}")
    axes[0, j].axis("off")
axes[0, 0].set_ylabel("Ground Truth", fontsize=10)

# Row 1: Corrupted
for j, t in enumerate(frame_indices):
    axes[1, j].imshow(corrupted[:, t].permute(1, 2, 0).numpy())
    axes[1, j].axis("off")
axes[1, 0].set_ylabel("Corrupted", fontsize=10)

# Rows 2+: Each lambda
for i, lam in enumerate(CFG["lambdas"]):
    model = ConditionalInpainter(base_ch=CFG["base_channels"]).to(DEVICE)
    model.load_state_dict(
        torch.load(all_results[lam]["model_path"], map_location=DEVICE, weights_only=True)
    )
    model.eval()
    with torch.no_grad():
        pred = model(corrupted_b, mask_b)

    for j, t in enumerate(frame_indices):
        axes[i+2, j].imshow(pred[0, :, t].cpu().permute(1, 2, 0).clamp(0,1).numpy())
        axes[i+2, j].axis("off")
    label = f"λ={lam}" + (" (baseline)" if lam == 0.0 else "")
    axes[i+2, 0].set_ylabel(label, fontsize=10)

plt.suptitle("Qualitative Comparison Across Guidance Strengths", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Error Maps Across Lambdas ----------

fig, axes = plt.subplots(len(CFG["lambdas"]), num_t, figsize=(4*num_t, 3*len(CFG["lambdas"])))

for i, lam in enumerate(CFG["lambdas"]):
    model = ConditionalInpainter(base_ch=CFG["base_channels"]).to(DEVICE)
    model.load_state_dict(
        torch.load(all_results[lam]["model_path"], map_location=DEVICE, weights_only=True)
    )
    model.eval()
    with torch.no_grad():
        pred = model(corrupted_b, mask_b)

    for j, t in enumerate(frame_indices):
        err = (pred[0, :, t].cpu() - clean[:, t]).abs().mean(0).numpy()
        im = axes[i, j].imshow(err, cmap="hot", vmin=0, vmax=0.3)
        axes[i, j].axis("off")
        if j == 0:
            axes[i, j].set_ylabel(f"λ={lam}", fontsize=10)
        if i == 0:
            axes[i, j].set_title(f"t={t}")

plt.suptitle("Per-Pixel Error Maps (Masked Regions)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Temporal Consistency Visualization ----------
# Show frame-to-frame difference magnitude across time for each lambda

fig, ax = plt.subplots(figsize=(10, 5))

# Ground truth temporal differences
gt_diffs = []
for t in range(CFG["num_frames"] - 1):
    d = (clean[:, t+1] - clean[:, t]).abs().mean().item()
    gt_diffs.append(d)
ax.plot(range(CFG["num_frames"]-1), gt_diffs, 'k--', linewidth=2, label="Ground Truth", marker='s')

for lam in CFG["lambdas"]:
    model = ConditionalInpainter(base_ch=CFG["base_channels"]).to(DEVICE)
    model.load_state_dict(
        torch.load(all_results[lam]["model_path"], map_location=DEVICE, weights_only=True)
    )
    model.eval()
    with torch.no_grad():
        pred = model(corrupted_b, mask_b)

    pred_diffs = []
    for t in range(CFG["num_frames"] - 1):
        d = (pred[0, :, t+1].cpu() - pred[0, :, t].cpu()).abs().mean().item()
        pred_diffs.append(d)
    ax.plot(range(CFG["num_frames"]-1), pred_diffs, 'o-', label=f"λ={lam}")

ax.set_xlabel("Frame Transition (t → t+1)")
ax.set_ylabel("Mean Absolute Frame Difference")
ax.set_title("Temporal Consistency: Frame-to-Frame Differences")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Output Diversity Analysis ----------
# Run multiple forward passes with different masks on the same clip
# to assess diversity of inpainting outputs

def measure_diversity(model, clean_clip, num_samples=5):
    """Measure diversity of inpainting outputs by generating multiple
    results with different masks and computing pairwise LPIPS distance."""
    model.eval()
    predictions = []

    for _ in range(num_samples):
        mask = generate_random_mask(CFG["num_frames"], CFG["img_size"], CFG["mask_ratio"])
        mask_t = torch.from_numpy(mask).unsqueeze(0).float()  # (1, T, H, W)
        corrupted = clean_clip * (1 - mask_t) + 0.5 * mask_t

        with torch.no_grad():
            pred = model(
                corrupted.unsqueeze(0).to(DEVICE),
                mask_t.unsqueeze(0).to(DEVICE)
            )[0].cpu()
        predictions.append(pred)

    # Pairwise LPIPS in masked regions (average over all frame pairs)
    distances = []
    for i in range(len(predictions)):
        for j in range(i + 1, len(predictions)):
            frame_dists = []
            for t in range(CFG["num_frames"]):
                p1 = predictions[i][:, t].unsqueeze(0).to(DEVICE) * 2 - 1
                p2 = predictions[j][:, t].unsqueeze(0).to(DEVICE) * 2 - 1
                with torch.no_grad():
                    d = lpips_model(p1, p2).item()
                frame_dists.append(d)
            distances.append(np.mean(frame_dists))

    return np.mean(distances)


print("Measuring output diversity...")
clean_sample = val_dataset[0][1]  # (3, T, H, W)
diversity_scores = {}

for lam in CFG["lambdas"]:
    model = ConditionalInpainter(base_ch=CFG["base_channels"]).to(DEVICE)
    model.load_state_dict(
        torch.load(all_results[lam]["model_path"], map_location=DEVICE, weights_only=True)
    )
    div = measure_diversity(model, clean_sample)
    diversity_scores[lam] = div
    print(f"  λ={lam}: Diversity (LPIPS) = {div:.4f}")

# Plot diversity
plt.figure(figsize=(8, 4))
plt.bar([str(l) for l in CFG["lambdas"]], [diversity_scores[l] for l in CFG["lambdas"]],
        color='steelblue', edgecolor='black')
plt.xlabel("λ (Guidance Strength)")
plt.ylabel("Pairwise LPIPS Distance")
plt.title("Output Diversity vs. Guidance Strength")
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Additional Failure Case Analysis ----------
# Show cases where high lambda causes over-smoothing

# Find clip with largest quality difference between low and high lambda
best_lam = min(eval_table, key=lambda l: eval_table[l]["lpips"])
worst_lam = max(eval_table, key=lambda l: eval_table[l]["lpips"])

print(f"Best LPIPS at λ={best_lam}: {eval_table[best_lam]['lpips']:.4f}")
print(f"Worst LPIPS at λ={worst_lam}: {eval_table[worst_lam]['lpips']:.4f}")

# Show multiple validation samples for best and worst lambda
fig, axes = plt.subplots(4, num_t, figsize=(4*num_t, 12))

sample_idx = 3  # different sample for variety
if sample_idx < len(val_dataset):
    corrupted2, clean2, mask2 = val_dataset[sample_idx]
    corrupted2_b = corrupted2.unsqueeze(0).to(DEVICE)
    mask2_b = mask2.unsqueeze(0).to(DEVICE)

    for row_idx, (lam, title) in enumerate([
        (None, "Ground Truth"),
        (None, "Corrupted"),
        (best_lam, f"Best (λ={best_lam})"),
        (worst_lam, f"Worst (λ={worst_lam})"),
    ]):
        for j, t in enumerate(frame_indices):
            if row_idx == 0:
                axes[row_idx, j].imshow(clean2[:, t].permute(1, 2, 0).numpy())
            elif row_idx == 1:
                axes[row_idx, j].imshow(corrupted2[:, t].permute(1, 2, 0).numpy())
            else:
                model = ConditionalInpainter(base_ch=CFG["base_channels"]).to(DEVICE)
                model.load_state_dict(
                    torch.load(all_results[lam]["model_path"], map_location=DEVICE, weights_only=True)
                )
                model.eval()
                with torch.no_grad():
                    pred2 = model(corrupted2_b, mask2_b)
                axes[row_idx, j].imshow(pred2[0, :, t].cpu().permute(1, 2, 0).clamp(0,1).numpy())
            axes[row_idx, j].axis("off")
            if j == 0:
                axes[row_idx, j].set_ylabel(title, fontsize=10)

    plt.suptitle("Success vs. Failure Case Comparison", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# ---------- Final Summary Table ----------

print("\n" + "="*90)
print("FINAL RESULTS SUMMARY")
print("="*90)
print(f"{'λ':>8} | {'PSNR ↑':>10} | {'SSIM ↑':>10} | {'LPIPS ↓':>10} | "
      f"{'TempCons ↓':>12} | {'Diversity':>10}")
print("-"*75)
for lam in CFG["lambdas"]:
    m = eval_table[lam]
    div = diversity_scores.get(lam, 0.0)
    tag = " *" if lam == best_lam else ""
    print(f"{lam:>8.2f} | {m['psnr']:>10.2f} | {m['ssim']:>10.4f} | "
          f"{m['lpips']:>10.4f} | {m['temporal_consistency']:>12.4f} | "
          f"{div:>10.4f}{tag}")
print("="*90)
print(f"* Best perceptual quality (LPIPS)")
print()
print("Key Findings:")
print("- Baseline (λ=0) optimizes purely for reconstruction accuracy")
print("- Moderate guidance (λ=0.01–0.1) often improves perceptual quality")
print("- Strong guidance (λ=0.5–1.0) may cause over-smoothing")
print("- Temporal consistency generally benefits from moderate guidance")

---
## Summary

### Task I — Unconditional Video Prior
We trained a 3D convolutional autoencoder on clean video clips as the unconditional prior $p_\theta(V)$. The model learns a latent representation of realistic video structure. Its encoder features and reconstruction error serve as the prior signal.

### Task II — Conditional Inpainting Baseline
A 3D U-Net with ConvLSTM temporal module takes corrupted video + mask as input and reconstructs the missing regions. Trained with masked L1 loss $\mathcal{L}_{\text{rec}}$ only.

### Task III — Prior-Guided Inpainting
We added two guidance mechanisms:
1. **Latent prior guidance**: L2 penalty between prior encoder features of predicted and clean videos
2. **Clip-level discriminator**: adversarial loss enforcing spatial and temporal realism

Combined objective: $\mathcal{L} = \mathcal{L}_{\text{rec}} + \lambda (0.5 \cdot \mathcal{L}_{\text{disc}} + 0.5 \cdot \mathcal{L}_{\text{latent}})$

### Task IV — Analysis
We evaluated across $\lambda \in \{0, 0.01, 0.1, 0.5, 1.0\}$ using:
- **Spatial**: PSNR, SSIM, LPIPS
- **Temporal**: Frame-difference consistency
- **Diversity**: Pairwise LPIPS across different mask samples

Results show the trade-off between reconstruction fidelity and perceptual realism as guidance strength increases.